# Release16 Drawdown-Drift Reward Recipe v1

Purpose: convert the `release16_trend_bc_slice_failure_drilldown_v1` handoff into a concrete short-training objective recipe. This notebook does not train, promote, or retain an actor. It screens drawdown-drift and trend-preservation reward patches against the official release16 failure daily paths so the next training run is not another same-family BC sweep.

Governance:
- Canonical baseline: `soxlbacktest_qlib_walkforward_baseline_v1`.
- Direct comparison baseline: `release16_trend_bc_slice_failure_drilldown_v1`.
- Closest historical success: `release16_risk_gated_state_head_objective_screen_v1` reduced MaxDD below 20% on the selected fold set but still missed the relative-return gate.
- Closest historical failure: `release16_downtrend_drawdown_objective_screen_v1` repaired drawdown but destroyed upside, confirming that hard de-risking alone is not enough.
- Role: diagnose / prepare short training screen.
- Changed layer: `reward` and `checkpoint_objective`; no runtime promotion.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Acceptance criteria for this screen: recipe captures at least 50% of drawdown-breach failure rows, at least 50% of trend-underparticipation rows, keeps overreach <= 25%, and emits an explicit reward patch for the next short fold-local BC/PPO screen.
- Final model gate remains stricter: official three-fold replay must reach relative return `>= 30pp`, MaxDD `< 20%`, and align with the SOXLBacktest qlib baseline before retention.


In [1]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.drawdown_drift_reward_screen import (
    DrawdownDriftRewardScreenConfig,
    render_drawdown_drift_reward_screen_review,
    screen_drawdown_drift_reward_recipes,
    summarize_drawdown_drift_reward_screen,
)

EXPERIMENT = "release16_drawdown_drift_reward_recipe_v1"
SOURCE_EXPERIMENT = "release16_trend_bc_slice_failure_drilldown_v1"
SOURCE_DAILY_EXPERIMENT = "release16_wf2020_trend_target_bc_official_replay_v1"
CLOSEST_SUCCESS = "release16_risk_gated_state_head_objective_screen_v1"
CLOSEST_FAILURE = "release16_downtrend_drawdown_objective_screen_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / SOURCE_EXPERIMENT
SOURCE_DAILY_CARD = ROOT / "runtime" / "model_selection" / "card" / SOURCE_DAILY_EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
REQUIRED_WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]

source_summary = json.loads((SOURCE_CARD / "summary.json").read_text(encoding="utf-8"))
source_candidate_metrics = pd.read_csv(SOURCE_DAILY_CARD / "bc_candidate_metrics.csv")
selected_daily_metrics = source_candidate_metrics[source_candidate_metrics["candidate"].eq("risk_trend_balanced")].copy()
if set(selected_daily_metrics["fold"]) != set(REQUIRED_WINDOWS):
    raise ValueError("selected release16 daily metrics do not cover all required windows")

source_readout = pd.DataFrame([
    {
        "source_experiment": SOURCE_EXPERIMENT,
        "source_decision": source_summary.get("decision"),
        "source_next_mechanism": source_summary.get("next_mechanism"),
        "source_min_relative_return_pp": source_summary.get("min_wf_relative_return_pp"),
        "source_worst_max_drawdown_pct": source_summary.get("worst_wf_max_drawdown_pct"),
        "selected_candidate": "risk_trend_balanced",
    }
])
display(source_readout)


,source_experiment,source_decision,source_next_mechanism,source_min_relative_return_pp,source_worst_max_drawdown_pct,selected_candidate
0,release16_trend_bc_slice_failure_drilldown_v1,release16_trend_bc_same_sweep_stopped_objectiv...,risk_state_drawdown_drift_checkpoint_objective,12.286463,26.568904,risk_trend_balanced


In [2]:
failure_type_by_fold = {
    "wf_2020": "joint_tradeoff",
    "wf_2022": "drawdown_control",
    "wf_2023_2024": "underparticipation",
}

gate_rows: list[dict[str, Any]] = []
daily_paths: dict[str, pd.DataFrame] = {}
daily_manifest: list[dict[str, str]] = []
for row in selected_daily_metrics.to_dict("records"):
    fold = str(row["fold"])
    daily_path = ROOT / str(row["daily_path"])
    daily = pd.read_csv(daily_path)
    label = f"release16_trend_selected__{fold}"
    daily_paths[label] = daily
    daily_manifest.append({"label": label, "fold": fold, "path": str(daily_path.relative_to(ROOT))})
    relative_return = float(row["relative_return"])
    max_dd = float(row["max_dd"])
    gate_rows.append(
        {
            "profile": "release16_trend_selected",
            "window": fold,
            "relative_return": relative_return,
            "max_dd": max_dd,
            "relative_gap": max(0.0, TARGET_RELATIVE_RETURN_PP - relative_return),
            "max_dd_gap": max(0.0, max_dd - TARGET_MAX_DD_PCT),
            "gate_pass": bool(relative_return >= TARGET_RELATIVE_RETURN_PP and max_dd < TARGET_MAX_DD_PCT),
            "failure_type": failure_type_by_fold[fold],
            "daily_path": str(daily_path.relative_to(ROOT)),
        }
    )

gate_gaps = pd.DataFrame(gate_rows).sort_values("window")
daily_manifest_frame = pd.DataFrame(daily_manifest).sort_values("fold")
gate_gaps.to_csv(CARD / "gate_gaps.csv", index=False)
daily_manifest_frame.to_csv(CARD / "daily_manifest.csv", index=False)
display(gate_gaps)


,profile,window,relative_return,max_dd,relative_gap,max_dd_gap,gate_pass,failure_type,daily_path
0,release16_trend_selected,wf_2020,12.286463,26.568904,17.713537,6.568904,False,joint_tradeoff,runtime/model_selection/card/release16_wf2020_...
1,release16_trend_selected,wf_2022,65.269018,24.711475,0.000000,4.711475,False,drawdown_control,runtime/model_selection/card/release16_wf2020_...
2,release16_trend_selected,wf_2023_2024,19.035720,14.803893,10.964280,0.000000,False,underparticipation,runtime/model_selection/card/release16_wf2020_...


In [3]:
config = DrawdownDriftRewardScreenConfig(
    target_relative_return_pp=TARGET_RELATIVE_RETURN_PP,
    target_max_dd_pct=TARGET_MAX_DD_PCT,
    min_drawdown_failure_capture=0.50,
    min_trend_underparticipation_capture=0.50,
    max_overreach_share=0.25,
)
scored = screen_drawdown_drift_reward_recipes(
    daily_paths,
    gate_gaps=gate_gaps,
    selected_candidate="release16_trend_selected",
    config=config,
)
summary_core = summarize_drawdown_drift_reward_screen(scored)
review = render_drawdown_drift_reward_screen_review(scored, summary_core, top_n=12)

scored.to_csv(CARD / "recipe_scores.csv", index=False)
(CARD / "review.md").write_text(review, encoding="utf-8")
(CARD / "selected_reward_patch.json").write_text(
    json.dumps(summary_core.get("reward_objective_patch", {}), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

display(scored.head(12))
display(Markdown(review))


,recipe,recipe_ready,reward_recipe_loss,drawdown_failure_capture,trend_underparticipation_capture,overreach_share,active_rows,drawdown_breach_rows,drawdown_captured_rows,trend_opportunity_rows,trend_captured_rows,pressure_burden,drawdown_penalty,drawdown_threshold_pct,position_threshold,trend_floor_penalty,trend_floor_position,trend_lookback,min_buyhold_return_pct,reward_objective_patch_json
0,dd1.5_thr18_pos0.05_tf1_floor0.25_lb21,True,1.718068,0.903226,1.0,0.0,121,62,56,45,45,0.859034,1.5,18.0,0.05,1.0,0.25,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
1,dd1.5_thr18_pos0.05_tf1_floor0.45_lb21,True,1.825739,0.903226,1.0,0.0,252,62,56,158,158,0.912869,1.5,18.0,0.05,1.0,0.45,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
2,dd1.5_thr18_pos0.05_tf1_floor0.35_lb21,True,1.829620,0.903226,1.0,0.0,173,62,56,92,92,0.914810,1.5,18.0,0.05,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
3,dd1.5_thr18_pos0.1_tf1_floor0.25_lb21,True,1.832139,0.500000,1.0,0.0,96,62,31,45,45,0.916070,1.5,18.0,0.10,1.0,0.25,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
4,dd1.5_thr18_pos0.1_tf1_floor0.45_lb21,True,1.885838,0.500000,1.0,0.0,227,62,31,158,158,0.942919,1.5,18.0,0.10,1.0,0.45,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
5,dd1.5_thr18_pos0.1_tf1_floor0.35_lb21,True,1.922455,0.500000,1.0,0.0,148,62,31,92,92,0.961228,1.5,18.0,0.10,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 1.5, ""drawdown_drif..."
6,dd2_thr18_pos0.05_tf1_floor0.45_lb21,True,1.990929,0.903226,1.0,0.0,252,62,56,158,158,0.995465,2.0,18.0,0.05,1.0,0.45,21,5.0,"{""drawdown_drift_penalty"": 2.0, ""drawdown_drif..."
7,dd2_thr18_pos0.1_tf1_floor0.45_lb21,True,2.022230,0.500000,1.0,0.0,227,62,31,158,158,1.011115,2.0,18.0,0.10,1.0,0.45,21,5.0,"{""drawdown_drift_penalty"": 2.0, ""drawdown_drif..."
8,dd2_thr18_pos0.05_tf1_floor0.25_lb21,True,2.062102,0.903226,1.0,0.0,121,62,56,45,45,1.031051,2.0,18.0,0.05,1.0,0.25,21,5.0,"{""drawdown_drift_penalty"": 2.0, ""drawdown_drif..."
9,dd2_thr18_pos0.05_tf1_floor0.35_lb21,True,2.070244,0.903226,1.0,0.0,173,62,56,92,92,1.035122,2.0,18.0,0.05,1.0,0.35,21,5.0,"{""drawdown_drift_penalty"": 2.0, ""drawdown_drif..."


# Drawdown-Drift Reward Screen

## Decision

- decision: `drawdown_drift_reward_recipe_ready_for_short_training_screen`
- selected_recipe: `dd1.5_thr18_pos0.05_tf1_floor0.25_lb21`
- training_allowed_next: `True`
- next_mechanism: `short_foldlocal_bc_or_ppo_training_screen`

## Candidate Ranking

| recipe | ready | loss | dd capture | trend capture | overreach | active rows |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| dd1.5_thr18_pos0.05_tf1_floor0.25_lb21 | True | 1.72 | 0.90 | 1.00 | 0.00 | 121 |
| dd1.5_thr18_pos0.05_tf1_floor0.45_lb21 | True | 1.83 | 0.90 | 1.00 | 0.00 | 252 |
| dd1.5_thr18_pos0.05_tf1_floor0.35_lb21 | True | 1.83 | 0.90 | 1.00 | 0.00 | 173 |
| dd1.5_thr18_pos0.1_tf1_floor0.25_lb21 | True | 1.83 | 0.50 | 1.00 | 0.00 | 96 |
| dd1.5_thr18_pos0.1_tf1_floor0.45_lb21 | True | 1.89 | 0.50 | 1.00 | 0.00 | 227 |
| dd1.5_thr18_pos0.1_tf1_floor0.35_lb21 | True | 1.92 | 0.50 | 1.00 | 0.00 | 148 |
| dd2_thr18_pos0.05_tf1_floor0.45_lb21 | True | 1.99 | 0.90 | 1.00 | 0.00 | 252 |
| dd2_thr18_pos0.1_tf1_floor0.45_lb21 | True | 2.02 | 0.50 | 1.00 | 0.00 | 227 |
| dd2_thr18_pos0.05_tf1_floor0.25_lb21 | True | 2.06 | 0.90 | 1.00 | 0.00 | 121 |
| dd2_thr18_pos0.05_tf1_floor0.35_lb21 | True | 2.07 | 0.90 | 1.00 | 0.00 | 173 |
| dd2_thr18_pos0.1_tf1_floor0.35_lb21 | True | 2.13 | 0.50 | 1.00 | 0.00 | 148 |
| dd2_thr18_pos0.1_tf1_floor0.25_lb21 | True | 2.15 | 0.50 | 1.00 | 0.00 | 96 |

## Readout

- This is a reward-objective readiness screen, not a performance backtest.
- A ready recipe may enter a short training screen, but cannot promote an actor or runtime overlay.


In [4]:
comparison_rows: list[dict[str, Any]] = []
for name in [SOURCE_EXPERIMENT, CLOSEST_SUCCESS, CLOSEST_FAILURE]:
    p = ROOT / "runtime" / "model_selection" / "card" / name / "summary.json"
    data = json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}
    comparison_rows.append(
        {
            "experiment": name,
            "decision": data.get("decision"),
            "next_mechanism": data.get("next_mechanism") or data.get("recommended_next_mechanism"),
            "min_wf_relative_return_pp": data.get("min_wf_relative_return_pp"),
            "worst_wf_max_drawdown_pct": data.get("worst_wf_max_drawdown_pct"),
            "max_relative_gap_to_target_pp": data.get("max_relative_gap_to_target_pp"),
            "max_drawdown_gap_to_target_pp": data.get("max_drawdown_gap_to_target_pp") or data.get("max_drawdown_gap_to_target_pct"),
            "training_allowed_signal": data.get("next_training_allowed_after_target_revision") or data.get("next_training_allowed_after_objective_screen") or data.get("training_allowed_next"),
        }
    )
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(CARD / "historical_objective_comparison.csv", index=False)
display(comparison)


,experiment,decision,next_mechanism,min_wf_relative_return_pp,worst_wf_max_drawdown_pct,max_relative_gap_to_target_pp,max_drawdown_gap_to_target_pp,training_allowed_signal
0,release16_trend_bc_slice_failure_drilldown_v1,release16_trend_bc_same_sweep_stopped_objectiv...,risk_state_drawdown_drift_checkpoint_objective,12.286463,26.568904,17.713537,6.568904,True
1,release16_risk_gated_state_head_objective_scre...,risk_gated_state_head_objective_screen_no_full...,train_nonconvex_risk_gated_state_head_checkpoi...,21.333130,16.652151,8.666870,NaN,True
2,release16_downtrend_drawdown_objective_screen_v1,release16_downtrend_drawdown_objective_drawdow...,repair_wf2020_trend_participation_without_losi...,-5.602809,17.693927,35.602809,NaN,None


In [5]:
ready = bool(summary_core.get("training_allowed_next", False))
selected_recipe = str(summary_core.get("selected_recipe", ""))
selected_patch = dict(summary_core.get("reward_objective_patch", {}))
selected_row = scored.iloc[0].to_dict() if not scored.empty else {}

next_mechanism = "short_foldlocal_bc_or_ppo_training_screen" if ready else "revise_reward_state_or_checkpoint_objective"
decision = (
    "release16_drawdown_drift_reward_recipe_ready_for_short_training_screen"
    if ready
    else "release16_drawdown_drift_reward_recipe_not_ready"
)

selected_min_relative = float(gate_gaps["relative_return"].min())
selected_worst_dd = float(gate_gaps["max_dd"].max())
max_relative_gap = float(gate_gaps["relative_gap"].max())
max_dd_gap = float(gate_gaps["max_dd_gap"].max())
selected_by_fold_summary = {
    str(row["window"]): {
        "selected_candidate": "release16_trend_selected",
        "selected_relative_return": float(row["relative_return"]),
        "selected_max_dd": float(row["max_dd"]),
        "selected_failure_type": str(row["failure_type"]),
        "relative_gap_to_target_pp": float(row["relative_gap"]),
        "drawdown_gap_to_target_pct": float(row["max_dd_gap"]),
    }
    for row in gate_gaps.to_dict("records")
}
metrics = [
    {
        "window": str(row["window"]),
        "relative_return_pct": float(row["relative_return"]),
        "max_dd_pct": float(row["max_dd"]),
        "relative_gap_to_target_pp": float(row["relative_gap"]),
        "drawdown_gap_to_target_pct": float(row["max_dd_gap"]),
    }
    for row in gate_gaps.to_dict("records")
]

artifact_paths = {
    "gate_gaps": str((CARD / "gate_gaps.csv").relative_to(ROOT)),
    "daily_manifest": str((CARD / "daily_manifest.csv").relative_to(ROOT)),
    "recipe_scores": str((CARD / "recipe_scores.csv").relative_to(ROOT)),
    "selected_reward_patch": str((CARD / "selected_reward_patch.json").relative_to(ROOT)),
    "historical_objective_comparison": str((CARD / "historical_objective_comparison.csv").relative_to(ROOT)),
    "review": str((CARD / "review.md").relative_to(ROOT)),
}

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "role": "diagnose_reward_objective_recipe",
    "decision": decision,
    "primary_blocker": "training_contract_ready" if ready else "diagnostic_no_promotion",
    "source_experiment": SOURCE_EXPERIMENT,
    "source_daily_experiment": SOURCE_DAILY_EXPERIMENT,
    "direct_comparison_baseline": SOURCE_EXPERIMENT,
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "closest_historical_success_case": CLOSEST_SUCCESS,
    "closest_historical_failure_case": CLOSEST_FAILURE,
    "why_not_duplicate": "This screen scores a joint drawdown-drift plus trend-preservation reward patch on official release16 failure paths; it does not tune BC class weights, hard de-risking thresholds, or reuse a static path blend.",
    "changed_layer": ["reward", "checkpoint_objective", "model_selection"],
    "target_window": REQUIRED_WINDOWS,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "acceptance_criteria": "Recipe must cover >=50% drawdown-breach rows and >=50% trend-underparticipation rows with <=25% overreach; final actor still requires official WF relative >=30pp and MaxDD <20%.",
    "accepted_tradeoff": "This can only unlock a short fold-local training screen; it cannot promote or retain an actor before official replay gates pass.",
    "hypothesis": "A joint drawdown-drift penalty plus trend-preservation floor can correct the release16 pattern where wf_2020/wf_2022 need de-risking but wf_2023_2024 needs participation restored.",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_next": ready,
    "short_training_screen_allowed_next": ready,
    "bc_screen_allowed_next": ready,
    "ppo_allowed_next": False,
    "target_contract_gate": ready,
    "bc_readiness_gate": ready,
    "reward_recipe_gate": ready,
    "next_mechanism": next_mechanism,
    "next_action": "run short fold-local BC/PPO training screen with selected reward patch, then official three-fold replay before retention review" if ready else next_mechanism,
    "selected_recipe": selected_recipe,
    "selected_reward_recipe_loss": float(summary_core.get("selected_reward_recipe_loss", 0.0) or 0.0),
    "drawdown_failure_capture": float(summary_core.get("drawdown_failure_capture", 0.0) or 0.0),
    "trend_underparticipation_capture": float(summary_core.get("trend_underparticipation_capture", 0.0) or 0.0),
    "overreach_share": float(summary_core.get("overreach_share", 0.0) or 0.0),
    "selected_active_rows": int(selected_row.get("active_rows", 0) or 0),
    "selected_drawdown_breach_rows": int(selected_row.get("drawdown_breach_rows", 0) or 0),
    "selected_drawdown_captured_rows": int(selected_row.get("drawdown_captured_rows", 0) or 0),
    "selected_trend_opportunity_rows": int(selected_row.get("trend_opportunity_rows", 0) or 0),
    "selected_trend_captured_rows": int(selected_row.get("trend_captured_rows", 0) or 0),
    "reward_objective_patch": selected_patch,
    "selected_by_fold": selected_by_fold_summary,
    "metrics": metrics,
    "target_relative_return_by_fold_pp": {window: TARGET_RELATIVE_RETURN_PP for window in REQUIRED_WINDOWS},
    "min_wf_relative_return_pp": selected_min_relative,
    "worst_wf_max_drawdown_pct": selected_worst_dd,
    "max_relative_gap_to_target_pp": max_relative_gap,
    "max_drawdown_gap_to_target_pp": max_dd_gap,
    "required_wf_windows": REQUIRED_WINDOWS,
    "artifact_paths": artifact_paths,
    "training_handoff": [
        {
            "decision": "ready" if ready else "blocked",
            "action": next_mechanism,
            "next_action": "run short fold-local BC/PPO training screen with selected reward patch, then official three-fold replay before retention review" if ready else next_mechanism,
            "source_experiment": SOURCE_EXPERIMENT,
        }
    ],
    "risk_notes": [
        "This screen uses existing failure daily paths and does not prove post-training performance.",
        "The selected patch is intentionally low-burden: dd penalty 1.5, threshold 18%, position threshold 0.05, trend floor 0.25.",
        "The next run must still pass official three-fold replay and retention gate before any actor is kept.",
    ],
}

(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "selection.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

conclusion_lines = [
    "# Release16 Drawdown-Drift Reward Recipe v1",
    "",
    f"Decision: `{decision}`.",
    "",
    f"- Selected recipe: `{selected_recipe}`.",
    f"- Training allowed next: `{ready}` for short fold-local screen only.",
    f"- Drawdown failure capture: `{summary['drawdown_failure_capture']:.2%}`.",
    f"- Trend-underparticipation capture: `{summary['trend_underparticipation_capture']:.2%}`.",
    f"- Overreach share: `{summary['overreach_share']:.2%}`.",
    f"- Current source path still fails retention: min relative `{selected_min_relative:.2f}` pp and worst MaxDD `{selected_worst_dd:.2f}%`.",
    f"- Next mechanism: `{next_mechanism}`.",
    "",
    "Conclusion: use this reward patch for a short fold-local BC/PPO training screen, then rerun official three-fold replay before any actor retention decision. This advances the model path beyond same-family BC weighting while keeping SOXLBacktest qlib as the comparison baseline.",
]
(CARD / "conclusion.md").write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")

display(Markdown("\n".join(conclusion_lines)))


# Release16 Drawdown-Drift Reward Recipe v1

Decision: `release16_drawdown_drift_reward_recipe_ready_for_short_training_screen`.

- Selected recipe: `dd1.5_thr18_pos0.05_tf1_floor0.25_lb21`.
- Training allowed next: `True` for short fold-local screen only.
- Drawdown failure capture: `90.32%`.
- Trend-underparticipation capture: `100.00%`.
- Overreach share: `0.00%`.
- Current source path still fails retention: min relative `12.29` pp and worst MaxDD `26.57%`.
- Next mechanism: `short_foldlocal_bc_or_ppo_training_screen`.

Conclusion: use this reward patch for a short fold-local BC/PPO training screen, then rerun official three-fold replay before any actor retention decision. This advances the model path beyond same-family BC weighting while keeping SOXLBacktest qlib as the comparison baseline.